# Lab 6: 4-PSK and 4-QAM

A complex baseband sample carries two independent degrees of freedom: its phase and its
magnitude, or equivalently its in-phase (I) and quadrature (Q) components. A modulation scheme
can put information in either one, or in a well-chosen combination of both. This lab works
through three variations on that idea, each placing four points in the complex plane and
asking you to recover, from noisy measurements, which of the four was sent.

The first two designs are constant-modulus: every constellation point sits on the same circle,
so all of the information lives in phase alone. You will design one such constellation aligned
with the coordinate axes, and a second obtained by rotating every point of the first by the
same fixed angle. Undisturbed by noise, a receiver should get identical performance from both —
but you will see over the course of the lab whether that holds up once real hardware, and the
biases it introduces, gets involved.

The third design drops the single-circle constraint and places its four points at the corners
of a square instead, letting the two bits of each symbol act on the I and Q axes independently.
This is the combined amplitude-and-phase case: two bits, two dimensions, one bit per dimension.

None of these designs are worth anything without a receiver that can read phase correctly in
the first place. Two independently clocked radios never share a common frequency or phase
reference; the raw samples that arrive carry both the transmitted symbol and a slow rotation
from that mismatch, and the rotation has to be measured and removed before a single decision
can be trusted. That measurement, alongside the residual DC bias every receiver carries
regardless of what's on the air, is where this lab starts.

## Hardware Setup

Configure both radios: sample rate, carrier frequency, transmit scaling, and a fixed manual
receive gain. These values stay fixed for the rest of the lab, with one exception — the
transmit gain, which you will sweep deliberately in the gain-sweep tasks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import adi
import time

fs = int(1.0e6)
fc = int(3100e6)
tx_scale = 2**14

tx_sdr = adi.Pluto("ip:192.168.2.1")
rx_sdr = adi.Pluto("ip:192.168.3.1")

tx_sdr.sample_rate = fs
tx_sdr.tx_lo = fc
tx_sdr.tx_rf_bandwidth = fs
tx_sdr.tx_hardwaregain_chan0 = -10

rx_sdr.sample_rate = fs
rx_sdr.rx_lo = fc
rx_sdr.rx_rf_bandwidth = fs
rx_sdr.gain_control_mode_chan0 = "manual"
rx_sdr.rx_hardwaregain_chan0 = 40
rx_sdr.rx_buffer_size = 30000

## Task 1: Calibrate the Receiver's DC Offset

Every SDR receiver reports a small, roughly constant complex bias even with nothing on the
air — local-oscillator leakage into the receive path, combined with small imperfections in the
analog front end. Left uncorrected, this bias shifts every sample you capture later by the same
fixed amount, and for the constellations in this lab, several of which place points exactly on
the coordinate axes, that shift lands squarely on top of the numbers you are trying to measure.

Measure it directly. With the transmitter idle, capture a burst of receive samples. The first
samples after a capture starts are typically still settling, so discard an initial stretch of
them and average only what remains. The result is your estimate of the receiver's DC offset, to
be subtracted from every capture for the rest of this lab.

In [ ]:
tx_sdr.tx_destroy_buffer()
time.sleep(0.2)

rx_idle_raw = rx_sdr.rx()   # TODO: capture a burst of RX samples with the transmitter idle
dc_offset = np.mean(rx_idle_raw[1000:])      # TODO: average the settled portion of rx_idle_raw (discard the
                                             #       initial transient samples first)
rx_sdr.rx_destroy_buffer()

print(f"DC offset estimate: {dc_offset.real:.2f} + {dc_offset.imag:.2f}j")

## Task 2: Calibrate Carrier Frequency and Phase Offset

Every constellation in this lab depends on the receiver correctly reading the *phase* of what
arrives, not just its magnitude. That is a problem, because the transmitter and receiver are two
separate radios, each running its own free-running local oscillator. Even nominally identical
hardware never agrees on frequency down to the last hertz, and that small mismatch shows up as a
continuous rotation: a sample's phase drifts steadily over the length of a capture, at a rate set
by the frequency offset between the two oscillators, on top of whatever fixed phase difference
happened to exist the moment the transmitter started.

Measure both quantities using a signal with no modulation to get in the way. Transmit an
unmodulated carrier — a constant complex value, held steady rather than switched between symbols
— and capture a burst of the result, discarding the initial settling stretch as before and
removing the DC offset from Task 1. Track how the phase of this calibration capture evolves from
one sample to the next: a steady, unchanging rate of change is exactly what a frequency offset
looks like, and fitting a straight line to the unwrapped phase across the burst hands you both
quantities at once — its slope gives you the frequency offset (convert from radians per sample
to hertz using the sample rate), and its intercept gives you the residual phase at the start of
the capture.

Keep both numbers. You will undo their effect on every later capture by multiplying against a
complex exponential built from them, before doing anything else with the samples.

In [ ]:
tx_sdr.tx_destroy_buffer()
time.sleep(0.2)

# A constant complex tone has no modulation, so its phase drift directly
# reveals the TX/RX frequency mismatch.
tx_cal_tone = (np.ones(20000, dtype=np.complex64) * tx_scale)

tx_sdr.tx_cyclic_buffer = True
tx_sdr.tx(tx_cal_tone)
time.sleep(0.5)

rx_cal_raw = rx_sdr.rx()
tx_sdr.tx_destroy_buffer()
rx_sdr.rx_destroy_buffer()

rx_cal = rx_cal_raw[1000:] - dc_offset

phase_unwrapped = np.unwrap(np.angle(rx_cal))
sample_index = np.arange(len(rx_cal))

# phase[n] = slope*n + intercept
slope, phase_hat = np.polyfit(sample_index, phase_unwrapped, 1)
cfo_hat = slope * fs / (2 * np.pi)

print(f"CFO estimate: {cfo_hat:.1f} Hz")
print(f"Phase offset estimate: {phase_hat:.3f} rad")

## Task 3: Design the 4-PSK Constellation

Design a four-point constellation:

- All four points at equal distance from the origin.
- Spaced 90° apart from one another.
- One point placed on the positive real (I) axis.

Assign each of the four points a unique 2-bit label using Gray coding, so that any two points
adjacent in phase differ in exactly one bit. Store the result in a form you can look up both
ways: from a 2-bit value to its transmit point, and later, from a received point's decision
back to the 2 bits it represents.

In [ ]:
PSK4_A_POINTS = np.array([
    1,
    1j,
    -1,
    -1j
], dtype=np.complex64)   # TODO: four equal-magnitude constellation points meeting the
                          #       spec above

PSK4_A_GRAY = np.array([
    [0, 0],
    [0, 1],
    [1, 1],
    [1, 0]
], dtype=np.uint8)      # TODO: 2-bit Gray-coded labels, in the same order as PSK4_A_POINTS

## Task 4: Build the Transmit Frame

Choose a frame length: enough information bits that, even at a fairly low error rate, you can
expect to see enough errors later to estimate the bit error rate reliably. On the order of a
hundred thousand bits is a reasonable target.

Generate that many random bits, group them into 2-bit symbols, and look up each symbol's
transmit point from the mapping you built in Task 3. Turn each complex symbol value into a
rectangular pulse by holding it constant for a fixed number of samples per symbol, and scale
the whole waveform so it uses a modest fraction of the transmitter's full output range rather
than driving the hardware at its limit. Hold on to the original bit sequence — you will need it
later to score errors.

Write this as a function of the symbol mapping rather than hardcoding Task 3's constellation
directly. You will reuse it, unchanged, for the other two constellations later in this lab.

In [ ]:
sps = 32

def build_tx_frame(points, gray_labels, n_bits):
    '''
    TODO: implement the procedure above for an arbitrary constellation. Return the
    transmitted bits, the transmitted symbols (integer indices into `points`), and the
    complex baseband waveform (complex64), scaled and pulse-shaped at `sps` samples per
    symbol.
    '''

    if n_bits <= 0 or n_bits % 2 != 0:
        raise ValueError("n_bits must be a positive even integer.")

    tx_bits = np.random.randint(
        0, 2, n_bits
    )

    bit_pairs = tx_bits.reshape(-1, 2)

    tx_symbols = np.array([
        np.where(
            np.all(gray_labels == pair, axis=1)
        )[0][0]
        for pair in bit_pairs
    ])

    tx_values = points[tx_symbols]

    tx_waveform = np.repeat(
        tx_values,
        sps
    )

    tx_waveform = (
        tx_waveform * tx_scale
    ).astype(np.complex64)

    return tx_bits, tx_symbols, tx_waveform


n_bits = 100000   # TODO: choose a frame length (see above); must be even

tx_bits_a, tx_symbols_a, tx_waveform_a = build_tx_frame(PSK4_A_POINTS, PSK4_A_GRAY, n_bits)

## Task 5: Transmit and Capture

Size the receive buffer to comfortably hold at least one full frame with room to spare. Send
the waveform using a cyclic transmit buffer, so the frame repeats continuously while you
capture, then grab a burst of receive samples. As in Tasks 1 and 2, the very start of a capture
is still settling, so discard an initial stretch of it.

Before doing anything else with the captured samples, subtract the DC offset from Task 1, then
undo the frequency and phase rotation estimated in Task 2 by multiplying against the matching
complex exponential. Every later step in this lab assumes both corrections have already been
applied.

Write this, too, as a function of the waveform being sent, so the same procedure can be reused
for the other two constellations without being retyped.

In [ ]:
def transmit_and_capture(tx_waveform, tx_gain_db=-10):
    '''
    TODO: implement the procedure above. Set tx_sdr's hardware gain to tx_gain_db, size
    rx_sdr's buffer to comfortably hold one frame, transmit tx_waveform cyclically, capture
    a burst, tear down both buffers, discard the initial transient, and apply the Task 1
    and Task 2 corrections. Return the calibrated, corrected complex samples.
    '''

    tx_sdr.tx_hardwaregain_chan0 = tx_gain_db

    rx_sdr.rx_buffer_size = (
        4 * len(tx_waveform) + 5000
    )

    tx_sdr.tx_destroy_buffer()
    rx_sdr.rx_destroy_buffer()

    tx_sdr.tx_cyclic_buffer = True
    tx_sdr.tx(tx_waveform)

    time.sleep(0.5)

    rx_raw = rx_sdr.rx()

    tx_sdr.tx_destroy_buffer()
    rx_sdr.rx_destroy_buffer()

    rx_calibrated = (
        rx_raw[1000:] - dc_offset
    )

    n = np.arange(len(rx_calibrated))

    phase_correction = (
        (2 * np.pi * cfo_hat / fs) * n
        + phase_hat
    )

    rx_corrected = (
        rx_calibrated *
        np.exp(-1j * phase_correction)
    )

    return rx_corrected.astype(np.complex64)


rx_a = transmit_and_capture(tx_waveform_a)

## Task 6: Visualize the Receiver Chain

From `rx_a`, pick one representative sample from the middle of each symbol's pulse, spaced by
your samples-per-symbol value, the same way a rectangular pulse's near-constant middle made
single-sample readings reasonable for amplitude. Keep both the full sample sequence and this
once-per-symbol sequence.

Produce a single figure with three panels, side by side.

1. A scatter plot of the once-per-symbol samples in the I/Q plane, with the four ideal
   constellation points from Task 3 marked on top, so you can see how tightly the received
   points cluster around where they should be.
2. A histogram of the once-per-symbol samples' phase angles, so you can see whether four
   distinct clusters are visible above the noise floor.
3. A plot of phase angle against sample index across the entire capture, so you can confirm
   the correction from Task 2 has left the phase flat over time rather than drifting.

Label every axis and give each panel a title.

In [ ]:
rx_center_a = rx_a[sps // 2::sps]   # TODO: one sample from the middle of every symbol's pulse in
                                    #       rx_a, sps samples apart
rx_phase_a = np.angle(rx_center_a)   # TODO: phase angle (radians) of every value in rx_center_a

plt.figure(figsize=(16, 5))

plt.subplot(1, 3, 1)
plt.scatter(
    rx_center_a.real,
    rx_center_a.imag,
    s=5,
    alpha=0.4,
    label="Received samples"
)
plt.scatter(
    PSK4_A_POINTS.real,
    PSK4_A_POINTS.imag,
    marker="x",
    s=100,
    linewidths=2,
    label="Ideal constellation"
)
plt.xlabel("In-Phase (Real)")
plt.ylabel("Quadrature (Imaginary)")
plt.title("4-PSK Constellation")
plt.grid()
plt.axis("equal")
plt.legend()   # TODO: scatter of rx_center_a in the I/Q plane, with PSK4_A_POINTS marked on top,
               #       title, axis labels, grid

plt.subplot(1, 3, 2)
plt.hist(rx_phase_a, bins=60)
plt.xlabel("Phase (radians)")
plt.ylabel("Count")
plt.title("Received Symbol Phase")
plt.grid()   # TODO: histogram of rx_phase_a, title, axis labels, grid

plt.subplot(1, 3, 3)
plt.plot(
    np.arange(len(rx_a)),
    np.angle(rx_a)
)
plt.xlabel("Sample Index")
plt.ylabel("Phase (radians)")
plt.title("Received Phase vs Sample Index")
plt.grid()   # TODO: phase angle of every sample in rx_a plotted against sample index, title,
             #       axis labels, grid

plt.tight_layout()
plt.show()

## Task 7: Symbol Decision and Bit Recovery

Using the four ideal constellation points from Task 3 as reference, decide which point each
value in your once-per-symbol sequence is closest to. Record that decision as a 2-bit symbol,
using the same Gray-coded mapping you built earlier, then concatenate the recovered symbols'
bits into one long sequence, in the same order the original bits were transmitted, so it can be
compared directly against your transmitted bit sequence.

Write this as a function of the constellation and once-per-symbol samples, rather than
hardcoding Task 3's points directly — you will reuse it for the other two constellations later
in this lab.

In [ ]:
def decide_symbols(rx_center, points, gray_labels):
    '''
    TODO: implement nearest-point decision for every value in rx_center against `points`,
    then map each decision back to bits via `gray_labels`, concatenated in transmission
    order. Return the recovered bits.
    '''

    distances = np.abs(
        rx_center[:, None] - points[None, :]
    )

    symbol_indices = np.argmin(
        distances,
        axis=1
    )

    rx_bits = gray_labels[
        symbol_indices
    ].flatten()

    return rx_bits


rx_bits_a = decide_symbols(rx_center_a, PSK4_A_POINTS, PSK4_A_GRAY)

## Task 8: Measure the Bit Error Rate

Compare your recovered bits against the transmitted bits, position by position, for the single
frame you just captured. Don't capture additional frames and pool them together — a single
frame's error count already tells you what you need, and averaging across frames would hide
the burst-to-burst variability you're trying to observe.

Count how many bit positions disagree, and report that count as a fraction of the total number
of bits in the frame. Write this as a function of the two bit sequences being compared, so you
can reuse it for the other two constellations later in this lab.

In [ ]:
def measure_ber(tx_bits, rx_bits):
    '''
    TODO: count positions where tx_bits and rx_bits differ, and return that count as a
    fraction of len(tx_bits).
    '''

    errors = np.count_nonzero(
        tx_bits != rx_bits
    )

    return errors / len(tx_bits)


ber_a = measure_ber(tx_bits_a, rx_bits_a)
print(f"4-PSK (0 deg) BER: {ber_a:.2e}")

## Task 9: Bit Error Rate versus Transmit Gain — 4-PSK (0°)

Repeat the transmit-capture-decide-score procedure from Tasks 4 through 8 at several different
transmit hardware gain settings, holding the receive gain fixed throughout, to see how the bit
error rate responds as the received signal strength changes relative to a fixed noise floor.

Wrap the full procedure in a function, so you can call it once per gain setting without
retyping it. Give that function a parameter for which decision function to use rather than
calling `decide_symbols` by name internally — later in this lab you will sweep a constellation
that uses a different one. Sweep a handful of gain values spanning from clearly error-free down
to clearly unusable. Plot the resulting bit error rate on a logarithmic vertical axis against
the gain values on a linear horizontal axis.


In [ ]:
def ber_vs_gain(points, gray_labels, n_bits, tx_gain_sweep_db, decide_fn=decide_symbols):
    '''
    TODO: for each gain in tx_gain_sweep_db, build a fresh frame (build_tx_frame), transmit
    and capture it at that gain (transmit_and_capture), decide symbols using decide_fn, and
    measure the BER (measure_ber). Return a list of BER values, one per gain.
    '''

    ber_values = []

    for gain in tx_gain_sweep_db:

        tx_bits, tx_symbols, tx_waveform = build_tx_frame(
            points,
            gray_labels,
            n_bits
        )

        rx = transmit_and_capture(
            tx_waveform,
            tx_gain_db=gain
        )

        rx_center = rx[sps // 2::sps]

        rx_center = rx_center[:len(tx_symbols)]

        rx_bits = decide_fn(
            rx_center,
            points,
            gray_labels
        )

        ber = measure_ber(
            tx_bits,
            rx_bits
        )

        ber_values.append(ber)

        print(f"TX gain = {gain} dB, BER = {ber:.2e}")

    return ber_values


tx_gain_sweep_db = [-30, -25, -20, -15, -10, -5, 0]   # TODO: a handful of TX gain settings to sweep
ber_results_a = ber_vs_gain(PSK4_A_POINTS, PSK4_A_GRAY, n_bits, tx_gain_sweep_db)

plt.figure(figsize=(7, 5))
plt.semilogy(
    tx_gain_sweep_db,
    ber_results_a,
    "o-"
)   # TODO: plt.semilogy(...) of ber_results_a vs. tx_gain_sweep_db, axis labels,
    #       grid, title
plt.xlabel("TX Gain (dB)")
plt.ylabel("Bit Error Rate (BER)")
plt.title("4-PSK (0 deg): BER vs TX Gain")
plt.grid(True, which="both")
plt.show()

## Task 10: Design a Rotated 4-PSK Constellation

Design a second four-point constellation with the same magnitude and spacing as Task 3 — four
points, equally spaced 90° apart — but rotate the whole arrangement by 30° so that none of its
points fall on either axis. Every point from Task 3 should now sit 30° further along the circle
than it did before.

Assign new 2-bit Gray-coded labels to these rotated points, following the same rule as Task 3:
any two points adjacent in phase should differ in exactly one bit. Store the mapping the same
way you did before, in both directions.

In [ ]:
PSK4_B_POINTS = (
    PSK4_A_POINTS *
    np.exp(1j * np.deg2rad(30))
).astype(np.complex64)   # TODO: PSK4_A_POINTS rotated by 30 degrees

PSK4_B_GRAY = PSK4_A_GRAY.copy()      # TODO: 2-bit Gray-coded labels, in the same order as PSK4_B_POINTS

## Task 11: Repeat for the Rotated Constellation

Run the same transmit, capture, and decision procedure you built in Tasks 4, 5, and 7 on this
new constellation, and measure its bit error rate as in Task 8. If those tasks were written as
functions of the constellation and its Gray coding rather than hardcoded to Task 3's specific
points, this should take no new code beyond calling what you already have with
`PSK4_B_POINTS` and `PSK4_B_GRAY` in place of `PSK4_A_POINTS` and `PSK4_A_GRAY`.

The frame length and buffer sizing from Task 4 and Task 5 carry over unchanged — this
constellation has the same number of points as the last one, so nothing about the frame's
size needs to be recomputed.

In [ ]:
tx_bits_b, tx_symbols_b, tx_waveform_b = build_tx_frame(PSK4_B_POINTS, PSK4_B_GRAY)   # TODO: build_tx_frame with
                                                                                        #       PSK4_B_POINTS, PSK4_B_GRAY

rx_b = transmit_and_capture(tx_waveform_b)                                       # TODO: transmit_and_capture(tx_waveform_b)
rx_center_b = rx_b[sps // 2::sps]                                               # TODO: one sample per symbol from rx_b,
                                                                                  #       as in Task 6

rx_center_b = rx_center_b[:len(tx_symbols_b)]

rx_bits_b = decide_symbols(rx_center_b, PSK4_B_POINTS, PSK4_B_GRAY)               # TODO: decide_symbols with
                                                                                   #       PSK4_B_POINTS, PSK4_B_GRAY
ber_b = measure_ber(tx_bits_b, rx_bits_b)                                         # TODO: measure_ber(tx_bits_b, rx_bits_b)

print(f"4-PSK (30 deg) BER: {ber_b:.2e}")

## Task 12: Bit Error Rate versus Transmit Gain — 4-PSK (30°)

Repeat the gain sweep from Task 9 for the rotated constellation, using the same gain values so
the two curves can be compared directly later. Reuse the `ber_vs_gain` function from Task 9
unchanged; only the constellation and its Gray coding should differ.

In [ ]:
ber_results_b = ber_vs_gain(PSK4_B_POINTS, PSK4_B_GRAY, n_bits, tx_gain_sweep_db)   # TODO: ber_vs_gain with PSK4_B_POINTS, PSK4_B_GRAY, n_bits,
                                                                                      #       and the same tx_gain_sweep_db from Task 9

plt.figure(figsize=(7, 5))
plt.semilogy(
    tx_gain_sweep_db,
    ber_results_b,
    "s-"
)   # TODO: plt.semilogy(...) of ber_results_b vs. tx_gain_sweep_db, axis labels,
    #       grid, title
plt.xlabel("TX Gain (dB)")
plt.ylabel("Bit Error Rate (BER)")
plt.title("4-PSK (30 deg): BER vs TX Gain")
plt.grid(True, which="both")
plt.show()

## Task 13: Design the 4-QAM Constellation

Design a third four-point constellation, but this time let each of the two bits act on one
coordinate axis independently: one bit decides the sign of the point's real part, the other
decides the sign of its imaginary part, so the four points land at the four sign combinations
of a fixed real and imaginary magnitude. Match the energy per symbol to the constellations you
designed in Tasks 3 and 10, so the comparison later in this lab isn't confounded by one design
simply transmitting more power than another.

Choose a 2-bit labeling consistent with that independent-bit structure, and store the mapping
in both directions as before.

In [ ]:
a = 1 / np.sqrt(2)

QAM4_POINTS = np.array([
    a + 1j*a,
    a - 1j*a,
    -a - 1j*a,
    -a + 1j*a
], dtype=np.complex64)   # TODO: four points, one bit sets the sign of the real part, the
                          #       other sets the sign of the imaginary part; match Tasks 3/10's
                          #       per-symbol energy

QAM4_GRAY = np.array([
    [0, 0],
    [0, 1],
    [1, 1],
    [1, 0]
], dtype=np.uint8)       # TODO: 2-bit labels consistent with that independent-bit structure

## Task 14: Decide by Axis, Not by Distance

Task 7's decision rule works by finding whichever of the four constellation points a received
sample sits closest to, and it would still work correctly here. But this constellation was
built with a special structure — each bit controls one coordinate axis, and the two axes act
independently of one another. Before reaching for a full nearest-point search, consider whether
that structure allows a shortcut: a way to decide each of the two bits directly from a single
coordinate of the received sample, without comparing it against every candidate point at all.

Implement that shortcut as a function of the received samples and the constellation's Gray
coding. You will use it, and check it against Task 7's general decision function, once you have
real samples to test it on.

In [ ]:
def decide_bits_by_axis(rx_center, points, gray_labels):
    '''
    TODO: implement the axis-based shortcut decision described above, using only the sign of
    each received sample's real and imaginary parts. Return the recovered bits, concatenated
    in the same order and format decide_symbols would produce.
    '''

    real_positive_bit = gray_labels[
        np.argmax(points.real)
    ][0]

    imag_positive_bit = gray_labels[
        np.argmax(points.imag)
    ][1]

    first_bits = np.where(
        rx_center.real >= 0,
        real_positive_bit,
        1 - real_positive_bit
    )

    second_bits = np.where(
        rx_center.imag >= 0,
        imag_positive_bit,
        1 - imag_positive_bit
    )

    return np.column_stack(
        (first_bits, second_bits)
    ).flatten()

## Task 15: Repeat for 4-QAM

Run the transmit and capture procedure from Tasks 4 and 5 on the 4-QAM constellation from
Task 13. Decide symbols two ways — using Task 7's general nearest-point function, and using
Task 14's axis-based shortcut — and confirm the two agree on every recovered bit before moving
on. Measure the bit error rate, using either one, as in Task 8.

In [ ]:
tx_bits_c, tx_symbols_c, tx_waveform_c = build_tx_frame(QAM4_POINTS, QAM4_GRAY)   # TODO: build_tx_frame with
                                                                                    #       QAM4_POINTS, QAM4_GRAY

rx_c = transmit_and_capture(tx_waveform_c)           # TODO: transmit_and_capture(tx_waveform_c)
rx_center_c = rx_c[sps // 2::sps]                     # TODO: one sample per symbol from rx_c, as in Task 6

rx_center_c = rx_center_c[:len(tx_symbols_c)]

rx_bits_c_general = decide_symbols(
    rx_center_c,
    QAM4_POINTS,
    QAM4_GRAY
)   # TODO: decide_symbols with QAM4_POINTS, QAM4_GRAY

rx_bits_c_axis = decide_bits_by_axis(
    rx_center_c,
    QAM4_POINTS,
    QAM4_GRAY
)   # TODO: decide_bits_by_axis with QAM4_POINTS, QAM4_GRAY

print("Decision functions agree:", np.array_equal(rx_bits_c_general, rx_bits_c_axis))

ber_c = measure_ber(tx_bits_c, rx_bits_c_axis)   # TODO: measure_ber(tx_bits_c, rx_bits_c_axis)
print(f"4-QAM BER: {ber_c:.2e}")

## Task 16: Bit Error Rate versus Transmit Gain — 4-QAM

Repeat the gain sweep from Tasks 9 and 12 for the 4-QAM constellation, using the same gain
values, so all three curves can be compared directly in the next task. Pass Task 14's axis-based
function in as `ber_vs_gain`'s decision function this time, rather than the default.

In [ ]:
ber_results_c = ber_vs_gain(
    QAM4_POINTS,
    QAM4_GRAY,
    n_bits,
    tx_gain_sweep_db,
    decide_fn=decide_bits_by_axis
)   # TODO: ber_vs_gain with QAM4_POINTS, QAM4_GRAY, n_bits, the
    #       same tx_gain_sweep_db from Task 9, and decide_fn=
    #       decide_bits_by_axis

plt.figure(figsize=(7, 5))
plt.semilogy(
    tx_gain_sweep_db,
    ber_results_c,
    "^-"
)   # TODO: plt.semilogy(...) of ber_results_c vs. tx_gain_sweep_db, axis labels,
    #       grid, title
plt.xlabel("TX Gain (dB)")
plt.ylabel("Bit Error Rate (BER)")
plt.title("4-QAM: BER vs TX Gain")
plt.grid(True, which="both")
plt.show()

## Task 17: Compare All Three

Plot all three bit error rate curves — from Tasks 9, 12, and 16 — on a single set of axes,
against the same transmit gain values, with a legend identifying each constellation. This is
the comparison the rest of the lab has been building toward.

In [ ]:
plt.figure(figsize=(7, 5))

plt.semilogy(
    tx_gain_sweep_db,
    ber_results_a,
    "o-",
    label="4-PSK (0 deg)"
)

plt.semilogy(
    tx_gain_sweep_db,
    ber_results_b,
    "s-",
    label="4-PSK (30 deg)"
)

plt.semilogy(
    tx_gain_sweep_db,
    ber_results_c,
    "^-",
    label="4-QAM"
)   # TODO: plt.semilogy(...) of ber_results_a, ber_results_b, and ber_results_c, each
    #       against tx_gain_sweep_db, with a legend, axis labels, grid, title

plt.xlabel("TX Gain (dB)")
plt.ylabel("Bit Error Rate (BER)")
plt.title("BER Comparison: 4-PSK vs 4-PSK (30 deg) vs 4-QAM")
plt.grid(True, which="both")
plt.legend()
plt.show()

## Background

**DC offset and LO leakage.** A small amount of a receiver's own local-oscillator signal
routinely couples into its receive path, and analog mixers and amplifiers carry their own
small imbalances. Together these show up as a constant complex offset added to every sample,
independent of whatever signal is actually arriving over the air. It doesn't average away on
its own, and unless it's measured and removed, it biases the center of every constellation you
capture afterward — a bias that lands most directly on any point whose ideal position happens
to sit close to the origin's neighborhood along the offset's own direction.

**Carrier frequency and phase offset.** Two independently clocked radios never share a
frequency reference exactly. If the transmitter's carrier and the receiver's local oscillator
differ by even a fraction of a hertz, every sample the receiver captures carries an extra phase
term that grows linearly with time — the received phase at sample $n$ is the transmitted phase
plus $2\pi \Delta f\, n / f_s + \phi_0$, where $\Delta f$ is the frequency offset and $\phi_0$ is
whatever phase difference existed the instant the transmitter started. An unmodulated tone makes
both of these directly measurable: its transmitted phase is constant, so any change in the
received phase over the capture is entirely due to $\Delta f$ and $\phi_0$, recoverable from a
linear fit to the unwrapped phase. Every symbol decision made anywhere in this lab implicitly
assumes this rotation has already been undone.

**Constant-modulus constellations and rotation invariance.** In additive Gaussian noise, the
noise added to a transmitted symbol is circularly symmetric — equally likely to point in any
direction in the complex plane. Rotating an entire constellation by a fixed angle rotates its
decision boundaries by exactly the same angle, leaving the geometry between every point and
every boundary completely unchanged. This is why the two constant-modulus designs earlier in
this lab should, in principle, deliver identical bit error rates at a given transmit gain: the
underlying decision problem is the same problem, just described in rotated coordinates. Any
difference you observe between them in practice reflects something the ideal noise model
doesn't capture — most plausibly, how each design's specific point positions interact with a
residual DC offset or some other hardware asymmetry that isn't itself rotationally symmetric.

**Why the square constellation decouples into two independent decisions.** When a constellation
places its four points at every combination of $\pm a$ on the real axis and $\pm a$ on the
imaginary axis, its decision boundaries are exactly the coordinate axes themselves. Deciding
which side of the imaginary axis a received sample falls on determines one bit completely,
regardless of where the sample sits on the real axis, and the same holds in reverse for the
other bit and the real axis. The two decisions never interact. This is the same idea as viewing
this constellation as two independent binary decisions carried on two orthogonal
dimensions — not a coincidence of this particular design, but a direct consequence of building
the constellation as the product of two independent sign choices in the first place. Neither of
the phase-based constellations earlier in the lab has this property: their decision boundaries
are diagonal lines that mix both coordinates together, so no single coordinate can be read off
on its own.

## Questions

1. Write down $E_s$ for each of this lab's three constellations in terms of the design
   parameters you chose, and confirm they are equal by construction. What is $E_b$ for each,
   given 2 bits per symbol?

2. Derive the maximum-likelihood decision rule for an $M$-ary constellation in AWGN, and show
   that for equally likely, equal-energy symbols it reduces to "decide the point closest in
   Euclidean distance." Then explain why, for the two constellations built in Tasks 3 and 10,
   this reduces further to "decide the point closest in phase," while for the constellation in
   Task 13, it reduces further still to two independent sign decisions.

3. Under ideal AWGN, use a union bound argument to show that the constellations from Tasks 3,
   10, and 13 should all have the same symbol error probability at a given $E_s/N_0$. If your
   Task 17 comparison shows a visible difference between them, what non-ideal effect in the
   hardware would you look at first, and why?

4. If your Task 2 frequency offset estimate were off by even a small amount, every symbol later
   in the frame would arrive rotated by an angle that grows with its position in the capture.
   Which of your Task 6 panels would reveal this first, and what would it look like there?

5. The DC offset measured in Task 1 is a fixed point in the I/Q plane, independent of the
   transmitted signal. For which of this lab's three constellations would a small residual DC
   offset be most damaging to symbol decisions, and why?

## Answer

1. For all three constellations, we designed the points with the same energy, \(E_s=1\). Since each symbol carries 2 bits, \(E_b=E_s/2=1/2\).
Therefore, all three have equal \(E_s\) and \(E_b\) by construction.

2. In AWGN, choose the constellation point with the smallest Euclidean distance from the received point. For 4-PSK, all points have the same amplitude, so this is the point with the closest phase. For 4-QAM, the decision can be made independently using the sign of I and Q.

3. All three constellations have the same \(E_s\) and the same minimum distance, so ideally they should have the same symbol error probability.
If Task 17 shows a difference, first check DC offset or IQ imbalance in the hardware. These imperfections can move or distort the received constellation.

4. The third panel of Task 6 (phase vs sample index) will show this first.
Instead of a stable/flat phase, the phase will show a gradually increasing or decreasing slope. This means the constellation is continuously rotating with time.

5. A residual DC offset would be most damaging to 4-QAM, because its decision boundaries are directly on the I and Q axes. Even a small shift can move points across these boundaries. Thus, DC offset can cause more symbol errors in 4-QAM.
